# Train a detection model on Google Colab

Training for hours makes a laptop very hot. This notebook runs detection-team's whole
training pipeline on Colab's free GPU instead, and saves the result to your Google Drive.

**How to use it**

1. *Runtime → Change runtime type →* pick a **GPU** (T4 is fine), then **Save**.
2. Check the settings in the next cell (the default trains `ball_v2`).
3. *Runtime → Run all.* Colab will ask once for permission to use your Google Drive.

**What it does:** clones the repo, downloads TORSO-21 (~5.7 GB), builds data-team's
train/val/test split, trains, evaluates on the held-out recordings, and copies the model
(`.pt`), its provenance (`.json`) and a visual eval report to your Drive.

**If Colab disconnects** (free sessions do, after a few hours): just *Run all* again. Training
progress is saved to your Drive after every epoch, and the notebook picks up where it stopped.

Expect roughly 5–10 minutes of setup, then about 1–2 hours of training.

In [ ]:
# ---- Settings ----------------------------------------------------------------
REPO = "https://github.com/Boiler-Soccer-Robot-Association-BSRA/CV.git"
BRANCH = "main"
# Which model to train. The dataset it needs is built automatically from its config:
#   ball_v2.yaml        ball only, data-team's split (already trained)
#   ball_v2_aug.yaml    same + data-team's augmentation (compare with ball_v2)
#   ball_v3_960.yaml    ball_v2 at 960 px (tiny balls; slower)
#   multiclass_v2.yaml  ball, robot, goalpost, landmark
CONFIG = "detection-team/configs/ball_v2.yaml"
DRIVE_DIR = "/content/drive/MyDrive/BSRA-CV"  # where results are saved

print("Will train:", CONFIG)
print("Wrong one? Change the CONFIG line above, then Runtime -> Run all again.")

## 1. Connect Google Drive and check the GPU

In [ ]:
import subprocess

from google.colab import drive

drive.mount("/content/drive")
gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip()
print(gpu or "No GPU! Runtime -> Change runtime type -> GPU, then Run all again.")
assert gpu, "This notebook needs a GPU runtime."

## 2. Get the code and install its dependencies

In [ ]:
%cd /content
!rm -rf CV && git clone --quiet --branch "$BRANCH" "$REPO" CV
%cd /content/CV
!git log --oneline -1
!pip install --quiet -r requirements.txt

## 3. Download TORSO-21 (real-world images)

Downloads the two zips directly. (data-team's `scripts/download.py` currently fetches the
data twice, ~11 GB, so it's skipped here until that's fixed.)

In [ ]:
%%bash
set -e
DEST=data-team/datasets/TORSO-21/reality
mkdir -p "$DEST"
for SPLIT in train test; do
  if [ ! -f "$DEST/$SPLIT/annotations.yaml" ]; then
    wget --quiet --continue -O "/content/$SPLIT.zip" "https://data.bit-bots.de/TORSO-21/reality/$SPLIT.zip"
    unzip -q -o "/content/$SPLIT.zip" -d "$DEST" && rm "/content/$SPLIT.zip"
  fi
  echo "$SPLIT: $(ls "$DEST/$SPLIT/images" | wc -l) images"
done

## 4. Build the dataset the config asks for

data-team's `split.py` (seed 42: same split on every computer), plus `augment.py` when the config
trains on `data_aug.yaml`. Old `_v1` configs still build detection's retired placeholder split.

In [ ]:
from pathlib import Path

import yaml

dataset = Path(yaml.safe_load(open(CONFIG))["dataset"]).as_posix()
multi = "multiclass" in dataset
classes = "ball robot goalpost landmark" if multi else "ball"
if "data-team/datasets/" in dataset:
    out_dir = "data-team/datasets/" + ("multiclass" if multi else "ball")
    !python data-team/scripts/split.py --classes $classes
    if dataset.endswith("data_aug.yaml"):
        !python data-team/scripts/augment.py $out_dir
else:  # retired placeholder (ball_v1 / multiclass_v1)
    !python detection-team/training/placeholder_dataset.py --classes $classes

## 5. Keep training progress on Drive

Training checkpoints go to `runs/train/` on your Drive, so a disconnect doesn't lose them.

In [ ]:
import os
from pathlib import Path

drive_runs = Path(DRIVE_DIR) / "runs" / "train"
drive_runs.mkdir(parents=True, exist_ok=True)
local_runs = Path("detection-team/runs")
local_runs.mkdir(parents=True, exist_ok=True)
if not (local_runs / "train").is_symlink():
    os.symlink(drive_runs, local_runs / "train")
print("Checkpoints will be saved in", drive_runs)

## 6. Train (or resume)

If this model is already on your Drive, training is skipped and only the evaluation below runs again.


In [ ]:
import shutil

import yaml

name = yaml.safe_load(open(CONFIG))["name"]
done = Path(DRIVE_DIR) / "models" / f"{name}.pt"
last = drive_runs / name / "weights" / "last.pt"
if done.exists():
    # Already trained (e.g. "Run all" pressed again after it finished): don't retrain
    # or resume it, which would overwrite the good model. Just evaluate it again.
    print(f"{name} is already trained ({done}). Skipping training.")
    print("To train it again, give the config a new name.")
    Path("detection-team/models").mkdir(parents=True, exist_ok=True)
    shutil.copy2(done, f"detection-team/models/{name}.pt")
    shutil.copy2(done.with_suffix(".json"), f"detection-team/models/{name}.json")
else:
    resume = "--resume" if last.exists() else ""
    print(f"{'Resuming' if resume else 'Starting'} {name}")
    !python detection-team/training/train.py "$CONFIG" $resume

## 7. Evaluate on the held-out recordings and save everything to Drive

In [ ]:
import shutil

data_yaml = yaml.safe_load(open(CONFIG))["dataset"]
data_yaml = str((Path(CONFIG).parent / data_yaml).resolve())
report = Path(DRIVE_DIR) / "reports" / f"{name}_test"
weights = f"detection-team/models/{name}.pt"
evaluate = "detection-team/training/eval.py"
!python "{evaluate}" "{weights}" "{data_yaml}" --split test --report-dir "{report}"

models = Path(DRIVE_DIR) / "models"
models.mkdir(parents=True, exist_ok=True)
for suffix in (".pt", ".json"):
    shutil.copy2(f"detection-team/models/{name}{suffix}", models)
for result in Path("detection-team/runs/eval").glob(f"{name}_*.json"):
    shutil.copy2(result, models)
print("Saved to", models)
print("Visual report:", report / "index.html", "(download the folder and open index.html)")